# EDAV Assignment — Exploratory Data Analysis and Visualization
**Dataset:** E-Commerce Sales Dataset (Synthetic)

This assignment covers:
- Data Loading and Inspection
- Data Cleaning and Preprocessing
- Exploratory Data Analysis (EDA)
- GroupBy and Aggregation
- Data Visualization

---
> **Instructions:** Answer all questions below. Write your code in the provided cells. Do not modify the question cells.

## Setup — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

print('Libraries imported successfully!')

## Setup — Create the Dataset
Run the cell below to generate the E-Commerce Sales dataset used throughout this assignment.

In [ ]:
np.random.seed(42)
n = 500

categories    = ['Electronics', 'Clothing', 'Books', 'Home & Kitchen', 'Sports']
regions       = ['North', 'South', 'East', 'West', 'Central']
payment_modes = ['Credit Card', 'Debit Card', 'UPI', 'Net Banking', 'Cash on Delivery']

data = {
    'OrderID'     : [f'ORD{str(i).zfill(4)}' for i in range(1, n + 1)],
    'CustomerAge' : np.random.randint(18, 65, n),
    'Category'    : np.random.choice(categories, n),
    'Quantity'    : np.random.randint(1, 10, n),
    'UnitPrice'   : np.round(np.random.uniform(50, 5000, n), 2),
    'Discount'    : np.round(np.random.uniform(0, 0.4, n), 2),
    'Region'      : np.random.choice(regions, n),
    'PaymentMode' : np.random.choice(payment_modes, n),
    'Rating'      : np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.10, 0.20, 0.35, 0.30]),
    'Returned'    : np.random.choice(['Yes', 'No'], n, p=[0.15, 0.85])
}

df = pd.DataFrame(data)

# Inject missing values
df.loc[np.random.choice(df.index, 20, replace=False), 'CustomerAge'] = np.nan
df.loc[np.random.choice(df.index, 15, replace=False), 'Rating']      = np.nan
df.loc[np.random.choice(df.index, 10, replace=False), 'Discount']    = np.nan

# Add derived column
df['TotalAmount'] = np.round(df['Quantity'] * df['UnitPrice'] * (1 - df['Discount'].fillna(0)), 2)

print(f'Dataset created: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

---
## Section 1: Data Inspection
### Q1. Display the first 10 rows and last 5 rows of the dataset.

In [ ]:
# First 10 rows
print('=== First 10 Rows ===')
display(df.head(10))

# Last 5 rows
print('=== Last 5 Rows ===')
display(df.tail(5))

### Q2. Display the shape, column names, data types, and a statistical summary of the dataset.

In [ ]:
print('Shape:', df.shape)
print('\nColumn Names:', df.columns.tolist())
print('\nData Types:')
print(df.dtypes)
print('\nStatistical Summary:')
display(df.describe(include='all'))

### Q3. Identify the total number of missing values in each column. Which column has the most missing values?

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Values': missing, 'Percentage (%)': missing_pct})
missing_df = missing_df[missing_df['Missing Values'] > 0].sort_values('Missing Values', ascending=False)

print('Missing Value Summary:')
display(missing_df)

print(f"\nColumn with most missing values: '{missing_df.index[0]}' ({missing_df['Missing Values'].iloc[0]} missing)")

---
## Section 2: Data Cleaning
### Q4. Fill missing values in `CustomerAge` with the median age, `Rating` with the mode, and `Discount` with 0. Verify no missing values remain.

In [ ]:
df['CustomerAge'].fillna(df['CustomerAge'].median(), inplace=True)
df['Rating'].fillna(df['Rating'].mode()[0], inplace=True)
df['Discount'].fillna(0, inplace=True)

print('Missing values after cleaning:')
print(df.isnull().sum())
print('\nAll nulls removed!' if df.isnull().sum().sum() == 0 else 'Some nulls remain.')

### Q5. Add a new column `AgeGroup` that categorizes customers into: `'Teen'` (18–24), `'Young Adult'` (25–34), `'Adult'` (35–49), `'Senior'` (50+). Display the value counts.

In [ ]:
bins   = [17, 24, 34, 49, 100]
labels = ['Teen', 'Young Adult', 'Adult', 'Senior']

df['AgeGroup'] = pd.cut(df['CustomerAge'], bins=bins, labels=labels)

print('AgeGroup Value Counts:')
print(df['AgeGroup'].value_counts())

### Q6. Create a new column `RevenueCategory` that labels orders as `'High'` if `TotalAmount > 10000`, `'Medium'` if between 3000 and 10000 (inclusive), and `'Low'` otherwise.

In [ ]:
def label_revenue(amount):
    if amount > 10000:
        return 'High'
    elif amount >= 3000:
        return 'Medium'
    else:
        return 'Low'

df['RevenueCategory'] = df['TotalAmount'].apply(label_revenue)

print('Revenue Category Distribution:')
print(df['RevenueCategory'].value_counts())

---
## Section 3: Exploratory Data Analysis
### Q7. Find the top 3 product categories by total revenue. Print the category names and their total revenue.

In [ ]:
top3_categories = df.groupby('Category')['TotalAmount'].sum().sort_values(ascending=False).head(3)

print('Top 3 Categories by Total Revenue:')
for rank, (cat, rev) in enumerate(top3_categories.items(), 1):
    print(f'  {rank}. {cat}: ₹{rev:,.2f}')

### Q8. Find the region with the highest return rate (i.e., percentage of orders where `Returned == 'Yes'`).

In [ ]:
return_rate = df.groupby('Region').apply(lambda x: (x['Returned'] == 'Yes').sum() / len(x) * 100).round(2)
return_rate.name = 'ReturnRate(%)'
return_rate = return_rate.sort_values(ascending=False)

print('Return Rate by Region:')
print(return_rate)
print(f"\nHighest Return Rate Region: {return_rate.index[0]} ({return_rate.iloc[0]}%)")

### Q9. Which payment mode is most preferred by customers aged 18–30? Display the count of each payment mode for this age group.

In [ ]:
young_customers = df[df['CustomerAge'] <= 30]
payment_pref = young_customers['PaymentMode'].value_counts()

print('Payment Mode Preference (Age 18–30):')
print(payment_pref)
print(f"\nMost Preferred: {payment_pref.index[0]} ({payment_pref.iloc[0]} orders)")

### Q10. Compute the average `Rating` per `Category` and sort in descending order.

In [ ]:
avg_rating = df.groupby('Category')['Rating'].mean().round(2).sort_values(ascending=False)

print('Average Rating by Category:')
print(avg_rating.to_string())

### Q11. Using `groupby`, create a pivot-style summary showing the `mean TotalAmount` for each combination of `Category` and `Region`.

In [ ]:
pivot = df.groupby(['Category', 'Region'])['TotalAmount'].mean().round(2).unstack()

print('Mean TotalAmount — Category vs Region:')
display(pivot)

### Q12. Find the top 5 most expensive single orders (highest `TotalAmount`). Display all columns for those orders.

In [ ]:
top5_orders = df.nlargest(5, 'TotalAmount')[['OrderID', 'Category', 'Quantity', 'UnitPrice', 'Discount', 'TotalAmount', 'Region']]

print('Top 5 Most Expensive Orders:')
display(top5_orders)

---
## Section 4: Data Visualization
### Q13. Plot a bar chart showing the total revenue by `Category`. Use different colors for each bar.

In [ ]:
cat_revenue = df.groupby('Category')['TotalAmount'].sum().sort_values(ascending=False)
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(cat_revenue.index, cat_revenue.values, color=colors, edgecolor='white', linewidth=0.8)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
            f'₹{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9)

ax.set_title('Total Revenue by Product Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Category', fontsize=11)
ax.set_ylabel('Total Revenue (₹)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

### Q14. Plot a pie chart showing the distribution of orders across `Region`.

In [ ]:
region_counts = df['Region'].value_counts()
explode = [0.05] * len(region_counts)

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    region_counts.values,
    labels=region_counts.index,
    autopct='%1.1f%%',
    explode=explode,
    startangle=140,
    colors=sns.color_palette('pastel'),
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for at in autotexts:
    at.set_fontsize(10)

ax.set_title('Order Distribution by Region', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Q15. Plot a histogram of `CustomerAge` with 15 bins. Add a vertical line for the mean age.

In [ ]:
mean_age = df['CustomerAge'].mean()

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df['CustomerAge'], bins=15, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(mean_age, color='crimson', linestyle='--', linewidth=2, label=f'Mean Age: {mean_age:.1f}')

ax.set_title('Distribution of Customer Age', fontsize=14, fontweight='bold')
ax.set_xlabel('Age', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Q16. Plot a grouped bar chart comparing the average `TotalAmount` by `Category` across `RevenueCategory` (High / Medium / Low).

In [ ]:
grouped = df.groupby(['Category', 'RevenueCategory'])['TotalAmount'].mean().unstack()

ax = grouped.plot(kind='bar', figsize=(11, 6), colormap='Set2', edgecolor='white', linewidth=0.8)
ax.set_title('Avg Total Amount by Category and Revenue Category', fontsize=13, fontweight='bold')
ax.set_xlabel('Product Category', fontsize=11)
ax.set_ylabel('Avg Total Amount (₹)', fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.legend(title='Revenue Category', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
plt.tight_layout()
plt.show()

### Q17. Plot a heatmap of the correlation matrix for all numeric columns.

In [ ]:
numeric_df = df.select_dtypes(include='number')
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5,
            square=True, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Q18. Plot a boxplot of `TotalAmount` for each `Category` to visualize the spread and outliers.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(data=df, x='Category', y='TotalAmount', palette='Set3', ax=ax,
            flierprops=dict(marker='o', markerfacecolor='crimson', markersize=4, alpha=0.6))

ax.set_title('Distribution of Total Amount by Category', fontsize=13, fontweight='bold')
ax.set_xlabel('Category', fontsize=11)
ax.set_ylabel('Total Amount (₹)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
plt.tight_layout()
plt.show()

### Q19. Plot a scatter plot of `UnitPrice` vs `TotalAmount`, colored by `Category`. Add a legend.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
palette = sns.color_palette('tab10', n_colors=df['Category'].nunique())

for i, (cat, group) in enumerate(df.groupby('Category')):
    ax.scatter(group['UnitPrice'], group['TotalAmount'],
               label=cat, color=palette[i], alpha=0.6, s=25, edgecolors='none')

ax.set_title('Unit Price vs Total Amount by Category', fontsize=13, fontweight='bold')
ax.set_xlabel('Unit Price (₹)', fontsize=11)
ax.set_ylabel('Total Amount (₹)', fontsize=11)
ax.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Q20. Plot a count plot showing the number of returned vs non-returned orders per `Category`.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.countplot(data=df, x='Category', hue='Returned', palette={'Yes': '#E74C3C', 'No': '#2ECC71'},
              edgecolor='white', ax=ax)

ax.set_title('Returned vs Non-Returned Orders by Category', fontsize=13, fontweight='bold')
ax.set_xlabel('Category', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.legend(title='Returned', fontsize=10)
plt.tight_layout()
plt.show()

---
## Section 5: Advanced Analysis
### Q21. Using `pivot_table`, compute the total revenue and average rating per `Region` and `Category`.

In [ ]:
pt = pd.pivot_table(df,
                    values=['TotalAmount', 'Rating'],
                    index='Region',
                    columns='Category',
                    aggfunc={'TotalAmount': 'sum', 'Rating': 'mean'},
                    fill_value=0).round(2)

print('Pivot Table — Total Revenue & Avg Rating per Region & Category:')
display(pt)

### Q22. Identify any outliers in `TotalAmount` using the IQR method. How many outlier orders exist? Print their count and the threshold values.

In [ ]:
Q1 = df['TotalAmount'].quantile(0.25)
Q3 = df['TotalAmount'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['TotalAmount'] < lower_bound) | (df['TotalAmount'] > upper_bound)]

print(f'Q1            : ₹{Q1:,.2f}')
print(f'Q3            : ₹{Q3:,.2f}')
print(f'IQR           : ₹{IQR:,.2f}')
print(f'Lower Bound   : ₹{lower_bound:,.2f}')
print(f'Upper Bound   : ₹{upper_bound:,.2f}')
print(f'\nTotal Outliers: {len(outliers)}')
display(outliers[['OrderID', 'Category', 'TotalAmount']].head())

### Q23. Compute the overall discount impact: what is the total revenue lost due to discounts?

In [ ]:
df['GrossAmount']    = df['Quantity'] * df['UnitPrice']
df['DiscountAmount'] = df['GrossAmount'] - df['TotalAmount']

total_gross    = df['GrossAmount'].sum()
total_net      = df['TotalAmount'].sum()
total_discount = df['DiscountAmount'].sum()

print(f'Total Gross Revenue  : ₹{total_gross:>15,.2f}')
print(f'Total Net Revenue    : ₹{total_net:>15,.2f}')
print(f'Revenue Lost (Disc.) : ₹{total_discount:>15,.2f}')
print(f'Discount Impact      : {total_discount / total_gross * 100:.2f}%')

### Q24. Rank the regions by total revenue using the `rank()` function. Display the result as a DataFrame sorted by rank.

In [ ]:
region_revenue = df.groupby('Region')['TotalAmount'].sum().reset_index()
region_revenue.columns = ['Region', 'TotalRevenue']
region_revenue['Rank'] = region_revenue['TotalRevenue'].rank(ascending=False).astype(int)
region_revenue = region_revenue.sort_values('Rank')

print('Region Revenue Ranking:')
display(region_revenue.to_string(index=False))

---
## Section 6: Summary
### Q25. Write a brief summary of your findings from the analysis above.

### Summary of Findings

**Dataset Overview:**
- The dataset contains 500 e-commerce orders across 5 product categories and 5 regions.
- Missing values were found in `CustomerAge`, `Rating`, and `Discount` columns — all handled via imputation.

**Revenue Insights:**
- *[Fill in which category had the highest revenue based on your output]*
- Discounts resulted in approximately *[fill in %]* revenue loss.

**Customer Behavior:**
- Younger customers (18–30) mostly prefer *[fill in payment mode from Q9]*.
- The *[fill in region]* region showed the highest return rate.

**Product Ratings:**
- *[Fill in category]* received the highest average rating.

**Conclusion:**
- The IQR analysis identified outlier orders with very high total amounts, mostly in *[fill in category]* products.
- The heatmap showed *[describe key correlation]* between features.

> *Complete this section after running all the cells above.*